# IEEE-CIS Fraud Detection: 비용 민감형 사기탐지 모델 개발
## Day 7. 확률 보정(Calibration) + Threshold Optimization

**목표**
- LightGBM(튜닝), RF(기본값) 두 최종 후보에 CalibratedClassifierCV 보정 적용
- 보정 전후 PR-AUC + Calibration 비교하여 최종 모델 선정
- 비용함수(FP/FN) 기반 Threshold Optimization

**진행 순서**
1. 데이터 + 변수 세트 불러오기
2. LightGBM best params 불러오기 (day6_tuning_results.pkl)
3. LightGBM(튜닝) + RF(기본값) 재학습
4. CalibratedClassifierCV 보정 적용
5. 보정 전후 PR-AUC + Calibration Gap 비교
6. 최종 모델 선정
7. Threshold Optimization

### 1. 라이브러리 임포트 및 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
sys.path.append("..")

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', 100)

# 데이터
df = pd.read_parquet("../data/processed/day3_final.parquet")

# LightGBM best params
with open("../data/processed/day6_tuning_results.pkl", 'rb') as f:
    tuning_results = pickle.load(f)

best_params_lgbm = tuning_results['best_params']['LightGBM']

# 변수 세트 재정의 (Day 4/6과 동일)
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'UID', 'UID_v2']
target = 'isFraud'
groups = df['UID']

categorical_cols = df.drop(columns=exclude_cols).select_dtypes(include='category').columns.tolist()
numeric_cols = df.drop(columns=exclude_cols).select_dtypes(include=[np.number]).columns.tolist()
binary_cols = [c for c in numeric_cols if df[c].nunique() <= 2]
scale_cols = [c for c in numeric_cols if c not in binary_cols]

X = df.drop(columns=exclude_cols + [target])
y = df[target]

print(f"shape: {df.shape}")
print(f"LightGBM best params: {best_params_lgbm}")